In [1]:
!pip install gradio plotly pandas openpyxl

In [2]:
import warnings
import gradio as gr
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.tree import DecisionTreeRegressor  
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore', category=UserWarning)

# ==========================================
# 1. 파일 경로 지정
# ==========================================
file_demand_score = r"C:\Users\hahas\Step5_Base_Demand_Score_List.xlsx"
file_full_data = r"C:\Users\hahas\Step5_119_Final_Integrated_Score.xlsx"
file_catalog = r"C:\Users\hahas\Step3_통합_보험데이터_최종_1개월_월환산완료.csv"

# 전역 변수 초기화
df_score = None
df_full = None
df_catalog = None
dt_model = None
le_age = None
le_gender = None 

try:
    print("⏳ 데이터 로드 및 ML 엔진 학습 중...")
    df_score = pd.read_excel(file_demand_score)
    df_full = pd.read_excel(file_full_data)
    
    try:
        df_catalog = pd.read_csv(file_catalog)
    except UnicodeDecodeError:
        df_catalog = pd.read_csv(file_catalog, encoding='cp949')
        
    # 💡 Decision Tree ML 예측 엔진 학습
    df_full = df_full.dropna(subset=['연령대', 'Gender', '최종_Total_수요점수'])
    le_age = LabelEncoder()
    le_gender = LabelEncoder()
    
    df_full['연령대_코드'] = le_age.fit_transform(df_full['연령대'].fillna('알수없음').astype(str))
    df_full['성별_코드'] = le_gender.fit_transform(df_full['Gender'].fillna('알수없음').astype(str))
    
    dt_model = DecisionTreeRegressor(max_depth=5, random_state=42)
    dt_model.fit(df_full[['연령대_코드', '성별_코드']], df_full['최종_Total_수요점수'])
    
    # 보험 담보 5대 테마 정규화
    groups = {
        '중증질환': ['질병사망', '80%이상', '질병입원', '해외질병'],
        '중대상해': ['상해사망', '상해후유장해', '상해입원', '해외상해', '구원자'],
        '근골격계': ['상해외래', '도수', '체외', '증식', 'MRI', 'MRA', '비급여주사'],
        '일반외상': ['상해외래', '상해입원', '해외상해'],
        '일반질환': ['질병외래', '질병입원', '해외질병']
    }
    
    for k, kw_list in groups.items():
        cols = [c for c in df_catalog.columns if any(x in c for x in kw_list)]
        df_catalog[f'Raw_{k}'] = df_catalog[cols].sum(axis=1) if cols else 0
        max_v = df_catalog[f'Raw_{k}'].max()
        df_catalog[f'Score_{k}'] = (df_catalog[f'Raw_{k}'] / max_v * 100) if max_v > 0 else 0
        
    df_catalog['최종보험료_num'] = pd.to_numeric(df_catalog['최종보험료'], errors='coerce').fillna(999999)
    df_catalog['추천_회사명'] = df_catalog['상품플랜'].apply(lambda x: str(x).split('_')[0] if '_' in str(x) else str(x).split(' ')[0])
    df_catalog['가입나이_int'] = pd.to_numeric(df_catalog['가입나이'], errors='coerce').fillna(-1).astype(int)
    df_catalog['성별_clean'] = df_catalog['성별'].astype(str).str.strip()
    
    print("✅ Decision Tree 모델 학습 및 데이터 준비 완료!")
    
except Exception as e:
    print(f"🚨 에러 발생: {e}")

# ==========================================
# 2. 분석 엔진 
# ==========================================
def get_safe_label(encoder, target_val, fallback_idx=0):
    target_str = str(target_val).strip()
    if target_str in encoder.classes_: 
        return target_str
    for cls in encoder.classes_:
        if target_str[:1] in str(cls) or target_str[:2] in str(cls): 
            return cls
    return encoder.classes_[fallback_idx]

def run_mcdm_v5_decision_tree(age, gender, priority):
    if dt_model is None:
        return go.Figure(), go.Figure(), pd.DataFrame(), go.Figure(), pd.DataFrame(), go.Figure()
        
    target_age_int = int(age)
    target_age_group = (target_age_int // 10) * 10
    target_age_decade = f"{target_age_group}대"
    
    target_gender_kor = gender.strip()
    target_gender_code = 'M' if gender == '남성' else 'F'
    
    safe_age = get_safe_label(le_age, target_age_decade)
    safe_gender = get_safe_label(le_gender, target_gender_code)
    
    # 💡 예측 모델 호출 (Decision Tree)
    input_df = pd.DataFrame(
        [[le_age.transform([safe_age])[0], le_gender.transform([safe_gender])[0]]], 
        columns=['연령대_코드', '성별_코드']
    )
    predicted_risk = dt_model.predict(input_df)[0]
    
    # 119 통계 기반 원그래프 가중치
    user_risk = df_full[
        (df_full['Gender'] == safe_gender) & 
        (df_full['연령대'] == safe_age) & 
        (~df_full['사고 및 질환 항목'].str.contains('자해|자살|정신|장애', na=False))
    ]
    
    kws = {
        '중증질환': ['암', '신생물', '심장', '뇌혈관'], 
        '중대상해': ['운수', '익사', '연기', '불'], 
        '근골격계': ['추락', '미끄러짐'], 
        '일반외상': ['가해', '타살', '중독', '노출'], 
        '일반질환': ['당뇨', '장염', '호흡', '내과', '질환']
    }
    
    counts = {
        k: user_risk[user_risk['사고 및 질환 항목'].str.contains('|'.join(v))]['최종_Total_수요점수'].sum() 
        for k, v in kws.items()
    }
    total = sum(counts.values()) if sum(counts.values()) > 0 else 1
    weights = {k: v / total for k, v in counts.items()}
    max_risk_key = max(weights, key=weights.get)
    
    # 데이터 매칭
    plans = df_catalog[
        (df_catalog['가입나이_int'] == target_age_int) & 
        (df_catalog['성별_clean'] == target_gender_kor)
    ].copy()
    
    if plans.empty:
        plans = df_catalog[
            (df_catalog['가입나이_int'] == target_age_group) & 
            (df_catalog['성별_clean'] == target_gender_kor)
        ].copy()
        
    if plans.empty:
        plans = df_catalog[df_catalog['성별_clean'] == target_gender_kor].copy()
        
    plans = plans.drop_duplicates(subset=['상품플랜'])
    
    # [Track 1] 고객 주관적 선택
    if priority == '보험료 최소화':
        user_top = plans.sort_values(by='최종보험료_num', ascending=True)
    elif priority == '중증 질환 특화':
        user_top = plans.sort_values(by=['Score_중증질환', '최종보험료_num'], ascending=[False, True])
    else:
        plans['User_Total'] = plans[[f'Score_{k}' for k in weights.keys()]].sum(axis=1)
        user_top = plans.sort_values(by='User_Total', ascending=False)
        
    # [Track 2] AI 추천 (1차 고도화 적용: 연령/성별 기반 보장 충실도 순)
    plans['AI_Valid'] = plans[f'Score_{max_risk_key}'].apply(lambda x: 1 if x > 0 else 0)
    ai_plans = plans[plans['AI_Valid'] == 1].copy()
    
    if ai_plans.empty: 
        ai_plans = plans.copy()
        
    # [수정됨] 보험료를 무시하고 순수하게 나와 맞는 보장 점수(Utility)만 산출
    ai_plans['AI_Utility'] = ai_plans.apply(lambda r: sum(r[f'Score_{k}'] * weights[k] for k in weights.keys()), axis=1)
    
    # [수정됨] 가격과 상관없이, 나에게 가장 필요한 보장이 잘 되어있는 순서대로 1위 선정
    ai_top = ai_plans.sort_values(by='AI_Utility', ascending=False)
    
    # ==========================================
    # 5. UI 컴포넌트 생성기
    # ==========================================
    fig_gauge = go.Figure(go.Indicator(
        mode="gauge+number", 
        value=predicted_risk,
        title={'text': "🎯 AI 예측 119 잠재 위험도", 'font': {'size': 18}},
        gauge={
            'axis': {'range': [0, 100]}, 
            'bar': {'color': "darkblue"}, 
            'steps': [
                {'range': [0, 50], 'color': "lightgreen"}, 
                {'range': [50, 100], 'color': "salmon"}
            ], 
            'threshold': {'line': {'color': "red", 'width': 4}, 'thickness': 0.75, 'value': 50}
        }
    ))
    fig_gauge.update_layout(height=280, margin=dict(t=50, b=10, l=10, r=10))
    
    fig_weight = go.Figure(data=[go.Pie(
        labels=list(weights.keys()), 
        values=list(weights.values()), 
        hole=.4,
        textinfo='label+percent', 
        marker=dict(colors=['#f87171', '#fb923c', '#fbbf24', '#34d399', '#60a5fa'])
    )])
    fig_weight.update_layout(title="📊 내 연령/성별 5대 리스크 비중", height=280, margin=dict(t=40, b=10, l=10, r=10))
    
    def create_table(df):
        if df.empty: 
            return pd.DataFrame()
        return pd.DataFrame([{
            '순위': f"{i}위", 
            '상품명': r['상품플랜'], 
            '월 보험료': f"{int(r['최종보험료_num']):,}원", 
            '중증': f"{int(r['Score_중증질환'])}점", 
            '상해': f"{int(r['Score_중대상해'])}점", 
            '근골격': f"{int(r['Score_근골격계'])}점", 
            '외상': f"{int(r['Score_일반외상'])}점", 
            '질환': f"{int(r['Score_일반질환'])}점"
        } for i, (_, r) in enumerate(df.iterrows(), 1)])
        
    def create_bar(df, title_prefix):
        if df.empty: 
            return go.Figure().update_layout(title="🚨 데이터 없음", height=250)
        
        row = df.iloc[0]
        scores = [row[f'Score_{k}'] for k in weights.keys()]
        colors = ['#ef4444' if s == 0 else '#3b82f6' for s in scores]
        
        fig = go.Figure(data=[go.Bar(
            x=list(weights.keys()), 
            y=scores, 
            marker_color=colors, 
            text=[f"{int(s)}점" for s in scores], 
            textposition='auto'
        )])
        fig.update_layout(
            title=f"{title_prefix} {row['상품플랜']} 보장 수준 (1위)", 
            yaxis_range=[0, 100], 
            height=250, 
            margin=dict(t=40, b=0, l=0, r=0)
        )
        return fig
        
    return fig_gauge, fig_weight, create_table(user_top), create_bar(user_top, "👤"), create_table(ai_top), create_bar(ai_top, "🤖")

# ==========================================
# 6. Gradio UI 레이아웃
# ==========================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🛡️Insurance Recommender")
    
    with gr.Row():
        age_in = gr.Number(label="나이", value=28, precision=0)
        gender_in = gr.Radio(["남성", "여성"], label="성별", value="여성")
        priority_in = gr.Dropdown(
            ['보험료 최소화', '보장 충실도 집중', '중증 질환 특화'],
            label="사용자 (나의) 최우선 순위 설정",
            value="보험료 최소화"
        )
        btn = gr.Button("비교 분석 실행", variant="primary")
        
    with gr.Row():
        with gr.Column():
            risk_gauge = gr.Plot()
        with gr.Column():
            risk_weights = gr.Plot()
            
    gr.Markdown("---")
    
    with gr.Row():
        with gr.Column():
            gr.Markdown("### 👤 사용자 선택")
            u_table = gr.Dataframe(interactive=False)
            u_plot = gr.Plot()
            
        with gr.Column():
            gr.Markdown("### 🤖 AI 추천")
            a_table = gr.Dataframe(interactive=False)
            a_plot = gr.Plot()
            
    gr.Markdown("""
    ---
    ### 📑 점수 산정 근거 (Basis for Calculation)
    * **👤 사용자 트랙:** 입력하신 우선순위만을 절대 기준으로 삼아 전체 상품을 정렬합니다. 스크롤을 내려 하위권의 보장 점수를 확인해 보세요.
    * **🤖 AI 트랙 (맞춤형 보장):** 119 통계에 따라 내 나이와 성별에서 가장 위험한 항목의 비중을 높게 반영하여, **비용과 상관없이 오직 내게 가장 필요한 보장이 충실하게 짜인 상품 순**으로 정렬하여 보여줍니다.
    """)
    
    btn.click(
        fn=run_mcdm_v5_decision_tree,
        inputs=[age_in, gender_in, priority_in],
        outputs=[risk_gauge, risk_weights, u_table, u_plot, a_table, a_plot]
    )

if __name__ == "__main__":
    demo.launch(inline=True, share=False)

⏳ 데이터 로드 및 ML 엔진 학습 중...
✅ Decision Tree 모델 학습 및 데이터 준비 완료!
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
